In [ ]:
import os
import warnings
import pandas as pd
import numpy as np

# ML Models & Metrics
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

data_path = os.path.join('data', 'all_months_features.csv')
df_raw = pd.read_csv(data_path)

In [2]:
# Chronological sorting & Target shifting (t + 1)
df = df_raw.copy()
df['month_idx'] = df.groupby('Product_Name').cumcount() + 1
df = df.sort_values(by=['Product_Name', 'month_idx']).reset_index(drop=True)

targets_base = ['Min_Price', 'Avg_Price', 'Max_Price']
targets_next = ['Min_Price_next', 'Avg_Price_next', 'Max_Price_next']

for base_col, next_col in zip(targets_base, targets_next):
    df[next_col] = df.groupby('Product_Name')[base_col].shift(-1)

df_clean = df.dropna(subset=targets_next).copy().reset_index(drop=True)

In [3]:
#  Separate Features and Targets
non_feature_cols = [
    'Product_Name', 'Category', 'Unit', 'unit_canonical', 'month_name',
    'bs_year', 'bs_month'
] + targets_base + targets_next

feature_cols = [col for col in df_clean.columns if col not in non_feature_cols]

X = df_clean[feature_cols].copy().apply(pd.to_numeric, errors='coerce').fillna(0)
y = df_clean[targets_next].copy().fillna(0)

In [ ]:
# Time-based split
train_mask = (df_clean['month_idx'] <= 7).values
valid_mask = (df_clean['month_idx'] == 8).values
test_mask  = (df_clean['month_idx'] == 9).values

X_train, y_train = X[train_mask], y[train_mask]
X_valid, y_valid = X[valid_mask], y[valid_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

# Find index for Avg_Price_next
avg_idx = targets_next.index('Avg_Price_next')

In [5]:
# RANDOM FOREST: Average Price Prediction
print(" RANDOM FOREST (Avg_Price_next) ")
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model = MultiOutputRegressor(rf_base)
rf_model.fit(X_train, y_train)

y_pred_rf_test = rf_model.predict(X_test)
actual_rf = y_test.iloc[:, avg_idx].values
pred_rf = y_pred_rf_test[:, avg_idx]

rf_mae = mean_absolute_error(actual_rf, pred_rf)
rf_rmse = np.sqrt(mean_squared_error(actual_rf, pred_rf))
rf_r2 = r2_score(actual_rf, pred_rf)
print(f"[RF Test] Avg_Price_next -> MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}, R²: {rf_r2:.3f}")

 RANDOM FOREST (Avg_Price_next) 
[RF Test] Avg_Price_next -> MAE: 44.29, RMSE: 82.62, R²: 0.561


In [6]:
# LIGHTGBM: Average Price Prediction

print("\nLIGHTGBM (Avg_Price_next)")
lgb_base = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.01, num_leaves=31, random_state=42, n_jobs=-1, verbose=-1)
lgb_model = MultiOutputRegressor(lgb_base)
lgb_model.fit(X_train, y_train)

y_pred_lgb_test = lgb_model.predict(X_test)
# Constraint enforcement (Min <= Avg <= Max)
y_pred_lgb_test[:, 0] = np.minimum(y_pred_lgb_test[:, 0], y_pred_lgb_test[:, 1])
y_pred_lgb_test[:, 2] = np.maximum(y_pred_lgb_test[:, 2], y_pred_lgb_test[:, 1])

actual_lgb = y_test.iloc[:, avg_idx].values
pred_lgb = y_pred_lgb_test[:, avg_idx]

lgb_mae = mean_absolute_error(actual_lgb, pred_lgb)
lgb_rmse = np.sqrt(mean_squared_error(actual_lgb, pred_lgb))
lgb_r2 = r2_score(actual_lgb, pred_lgb)
print(f"[LightGBM Test] Avg_Price_next -> MAE: {lgb_mae:.2f}, RMSE: {lgb_rmse:.2f}, R²: {lgb_r2:.3f}")


LIGHTGBM (Avg_Price_next)
[LightGBM Test] Avg_Price_next -> MAE: 37.17, RMSE: 74.76, R²: 0.640


In [ ]:
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

optuna.logging.set_verbosity(optuna.logging.WARN)

# Time-based Split

train_mask = (df_clean['month_idx'] <= 7).values
valid_mask = (df_clean['month_idx'] == 8).values
test_mask  = (df_clean['month_idx'] == 9).values

X_train = X[train_mask]
y_train = y[train_mask]

X_valid = X[valid_mask]
y_valid = y[valid_mask]

X_test = X[test_mask]
y_test = y[test_mask]

# Target index for Avg_Price_next
avg_idx = targets_next.index("Avg_Price_next")

# Optuna Objective Function

def objective(trial):

    params = {
        "objective": "regression_l1",
        "num_leaves": trial.suggest_int("num_leaves", 15, 63),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.1, log=True
        ),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "n_estimators": trial.suggest_int(
            "n_estimators", 200, 1000
        ),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    model = MultiOutputRegressor(
        lgb.LGBMRegressor(**params)
    )

    model.fit(X_train, y_train)

    y_pred_valid = model.predict(X_valid)

    mae = mean_absolute_error(
        y_valid.iloc[:, avg_idx],
        y_pred_valid[:, avg_idx]
    )

    return mae

# Run Optuna

study = optuna.create_study(direction="minimize")

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)

print("\n========== BEST PARAMETERS ==========")
print(study.best_params)

print(f"\nBest Validation MAE : {study.best_value:.4f}")

# Train Final Model (Train + Validation)


best_params = study.best_params.copy()

best_params.update({
    "objective": "regression_l1",
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1
})

X_train_final = pd.concat(
    [X_train, X_valid],
    axis=0
)

y_train_final = pd.concat(
    [y_train, y_valid],
    axis=0
)

final_model = MultiOutputRegressor(
    lgb.LGBMRegressor(**best_params)
)

final_model.fit(
    X_train_final,
    y_train_final
)
# Final Test Evaluation

y_pred_final = final_model.predict(X_test)

actual = y_test.iloc[:, avg_idx].values
pred = y_pred_final[:, avg_idx]

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
r2 = r2_score(actual, pred)

print("\n========== FINAL TEST PERFORMANCE ==========")
print(f"Target : Avg_Price_next")
print(f"MAE    : {mae:.2f}")
print(f"RMSE   : {rmse:.2f}")
print(f"R²     : {r2:.3f}")

Best trial: 15. Best value: 31.77: 100%|██████████| 30/30 [00:48<00:00,  1.61s/it]  



--- BEST OPTUNA PARAMETERS ---
{'num_leaves': 19, 'learning_rate': 0.045880968456598135, 'subsample': 0.6388499541386699, 'colsample_bytree': 0.7780228457392366, 'reg_alpha': 1.2316594989479133, 'reg_lambda': 1.081071502041687, 'n_estimators': 908}
Best Test MAE (Avg_Price_next): 31.7700

[Optimized LightGBM Test] Avg_Price_next -> MAE: 31.77, RMSE: 67.56, R²: 0.706


In [8]:
# Diagnostic code to check residuals and analyze errors by category/product
import pandas as pd
import numpy as np

# Create a diagnostic dataframe for the test set
diagnostic_df = pd.DataFrame({
    'Product_Name': df_clean.loc[test_mask, 'Product_Name'].values,
    'Category': df_clean.loc[test_mask, 'Category'].values,
    'Actual_Avg': actual_final_avg,
    'Predicted_Avg': pred_final_avg
})

# Calculate error metrics
diagnostic_df['Error'] = diagnostic_df['Actual_Avg'] - diagnostic_df['Predicted_Avg']
diagnostic_df['Absolute_Error'] = np.abs(diagnostic_df['Error'])
diagnostic_df['Squared_Error'] = diagnostic_df['Error'] ** 2

# 1. View top 10 worst predictions (highest absolute error)
print("--- Top 10 Worst Predictions ---")
worst_predictions = diagnostic_df.sort_values(by='Absolute_Error', ascending=False).head(10)
print(worst_predictions[['Product_Name', 'Category', 'Actual_Avg', 'Predicted_Avg', 'Absolute_Error']])

# 2. Check average error grouped by Category to find problematic sectors
print("\n--- Error Breakdown by Category ---")
category_errors = diagnostic_df.groupby('Category').agg(
    Mean_Absolute_Error=('Absolute_Error', 'mean'),
    Root_Mean_Squared_Error=('Squared_Error', lambda x: np.sqrt(np.mean(x))),
    Count=('Product_Name', 'count')
).sort_values(by='Mean_Absolute_Error', ascending=False)
print(category_errors)

--- Top 10 Worst Predictions ---
        Product_Name   Category  Actual_Avg  Predicted_Avg  Absolute_Error
2            Avocado      fruit      757.61     375.417522      382.192478
26             Lemon      fruit      363.04     211.025406      152.014594
28             Mango      fruit      198.26     339.212590      140.952590
29          Mushroom  vegetable      219.35     127.534408       91.815592
5       Bitter_Gourd  vegetable       65.00     129.678409       64.678409
24  Khasa_Garlic_Dry  vegetable      233.70     292.794167       59.094167
17           Cow_Pea  vegetable       91.30     140.942940       49.642940
42      Smooth_Gourd  vegetable       78.26     127.079058       48.819058
33            Orange      fruit      209.57     164.555543       45.014457
30             Okara  vegetable       85.22     127.850693       42.630693

--- Error Breakdown by Category ---
           Mean_Absolute_Error  Root_Mean_Squared_Error  Count
Category                                  